# Fusion Design Lab — GRPO Training

Train an LLM to optimize stellarator fusion reactor designs using **GRPO** (Group Relative Policy Optimization) with **HF TRL**.

The agent interacts with a constrained optimization environment where it adjusts 4 geometric knobs of a stellarator boundary, aiming to **minimize max elongation** while satisfying 3 hard physics constraints:
- `aspect_ratio ≤ 4.0`
- `average_triangularity ≤ -0.5`
- `abs(edge_iota_over_nfp) ≥ 0.3`

Each episode has **6 evaluations** budgeted. The notebook now trains on the same live **low-fidelity environment contract** used by the repo runtime: `run`, `restore_best`, and explicit terminal `submit` all stay on the same verifier surface. Higher-fidelity checks live outside the notebook reward loop.

**Environment deployed at**: https://creativeengineer-fusion-design-lab.hf.space

**Runtime**: Select GPU via `Runtime > Change runtime type`. The notebook automatically uses `fp16` on T4/V100-class GPUs and `bf16` on Ampere-or-newer GPUs.

## 1. Install Dependencies

In [ ]:
%%capture
import importlib.util
import os
import shutil
import subprocess
import sys


def run_checked(command: list[str]) -> None:
    subprocess.run(command, check=True)


def maybe_install_build_deps() -> None:
    if sys.platform != "linux":
        return
    apt_get = shutil.which("apt-get")
    if apt_get is None or os.geteuid() != 0:
        return
    run_checked([apt_get, "update", "-qq"])
    run_checked(
        [
            apt_get,
            "install",
            "-y",
            "-qq",
            "cmake",
            "ninja-build",
            "g++",
            "gfortran",
            "libnetcdf-dev",
            "libnetcdff-dev",
        ]
    )


def ensure_pip() -> None:
    if importlib.util.find_spec("pip") is None:
        run_checked([sys.executable, "-m", "ensurepip", "--upgrade"])


def install_python_deps() -> None:
    ensure_pip()
    run_checked(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "trl==0.29.0",
            "peft>=0.15.0,<1.0",
            "bitsandbytes>=0.45.0,<1.0",
            "datasets>=3.0.0,<4.0",
            "matplotlib>=3.9.0,<4.0",
            "accelerate>=1.3.0,<2.0",
        ]
    )


maybe_install_build_deps()
install_python_deps()

if importlib.util.find_spec("torch") is None:
    raise RuntimeError(
        "PyTorch is not installed in this kernel. Use a CUDA-enabled Colab runtime "
        "or a Northflank PyTorch GPU notebook image before running this notebook."
    )

## 2. Load Model with LoRA

In [ ]:
import importlib
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Qwen/Qwen3.5-4B"
MAX_SEQ_LENGTH = 2048

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires a CUDA GPU runtime.")
gpu_major, _ = torch.cuda.get_device_capability()
use_bf16 = gpu_major >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

attn_impl = "flash_attention_2" if importlib.util.find_spec("flash_attn") else "sdpa"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto",
    attn_implementation=attn_impl,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()
dtype_name = "bf16" if use_bf16 else "fp16"
print(f"Model loaded: {MODEL_NAME} (attn: {attn_impl}, dtype: {dtype_name})")

## 3. Setup Stellarator Environment

Install the environment from the checked-out Fusion Design Lab repository when it is available in the runtime. If the notebook is running in a fresh Colab session, clone the public repo first and then install it in editable mode. This keeps the notebook bound to the same `server/environment.py` Reward V2 code and `server/physics.py` verifier code that ship with the notebook, instead of a potentially stale deployment copy.

In [ ]:
%%capture
from pathlib import Path
from typing import Final

REPO_URL = "https://github.com/jungdaesuh/fusion-design-lab.git"
EXPECTED_REPO_FILES: Final[tuple[str, ...]] = (
    "pyproject.toml",
    "server/environment.py",
    "server/physics.py",
    "fusion_lab/models.py",
    "fusion_lab/llm_agent.py",
    "training/notebooks/fusion_design_lab_training.ipynb",
)


def _is_valid_repo_root(candidate: Path) -> bool:
    return candidate.is_dir() and all((candidate / item).exists() for item in EXPECTED_REPO_FILES)


def resolve_repo_root() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/fusion-design-lab"),
        Path("/home/jovyan/fusion-design-lab"),
        Path.home() / "fusion-design-lab",
    ]
    for candidate in candidates:
        if _is_valid_repo_root(candidate):
            return candidate.resolve()

    target = (
        Path("/content/fusion-design-lab")
        if "google.colab" in sys.modules
        else Path.home() / "fusion-design-lab"
    )
    if not target.exists():
        subprocess.run(["git", "clone", REPO_URL, str(target)], check=True)
    if not _is_valid_repo_root(target):
        raise RuntimeError(
            "Could not locate a complete fusion-design-lab repository at {target}.".format(
                target=target
            )
        )
    return target.resolve()


REPO_ROOT = resolve_repo_root()
os.chdir(REPO_ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], check=True)

In [ ]:
import inspect
import json
import os
from pathlib import Path
from typing import Final

from fusion_lab.llm_agent import (
    RUN_DIRECTIONS,
    RUN_MAGNITUDES,
    RUN_PARAMETERS,
    build_messages,
    parse_action_plan,
    run_episode_with_actions,
)
from fusion_lab.models import StellaratorAction
from server.contract import RESET_SEEDS
from server.environment import BUDGET, StellaratorEnvironment
from server.physics import evaluate_boundary

RUN_ACTION_SPECS: Final[list[dict[str, str]]] = [
    {"intent": "run", "parameter": p, "direction": d, "magnitude": m}
    for p in RUN_PARAMETERS
    for d in RUN_DIRECTIONS
    for m in RUN_MAGNITUDES
]

AVAILABLE_ACTIONS: Final[list[dict[str, str]]] = RUN_ACTION_SPECS + [
    {"intent": "restore_best"},
]
# Quick smoke test
env = StellaratorEnvironment()
obs = env.reset(seed=0)
print(
    f"Environment ready. Initial score: {obs.p1_score:.4f}, feasibility: {obs.p1_feasibility:.4f}"
)
print(f"Budget: {obs.budget_remaining}, Constraints satisfied: {obs.constraints_satisfied}")


def _assert_expected_source(source: str | Path, *, expected: Path, label: str) -> Path:
    source_path = Path(source or "").resolve()
    if source_path.name != expected.name:
        raise RuntimeError(f"Expected {label} to come from {expected}, got {source_path}")
    if source_path != expected.resolve():
        raise RuntimeError(
            f"Expected {label} to come from {expected}, got {source_path}. This indicates an environment or module path mismatch."
        )
    return source_path


reward_source = _assert_expected_source(
    inspect.getsourcefile(StellaratorEnvironment._compute_reward_breakdown),
    expected=REPO_ROOT / "server" / "environment.py",
    label="Reward V2",
)
verifier_source = _assert_expected_source(
    inspect.getsourcefile(evaluate_boundary),
    expected=REPO_ROOT / "server" / "physics.py",
    label="Verifier",
)
print(f"Reward source bound to: {reward_source}")
print(f"Verifier source bound to: {verifier_source}")


def render_generation_prompt(observation):
    return tokenizer.apply_chat_template(
        list(build_messages(observation)),
        tokenize=False,
        add_generation_prompt=True,
    )

## 4. Prompt Template & Action Parsing

Each training sample is a prompt describing the stellarator task and initial state. The model generates a plan of actions to optimize the design.

In [ ]:
# Shared helper smoke test
env = StellaratorEnvironment()
obs = env.reset(seed=0)
prompt = render_generation_prompt(obs)
print(prompt[:500])
print("...")

sample_completion = json.dumps(
    [
        {
            "intent": "run",
            "parameter": "triangularity_scale",
            "direction": "increase",
            "magnitude": "small",
        },
        {"intent": "submit"},
    ]
)
print(parse_action_plan(sample_completion))

## 5. Training Dataset

Create prompts from all 3 reset seeds. Each prompt is an initial observation that the model must optimize.

In [ ]:
from datasets import Dataset

prompts = []
for seed_idx in range(len(RESET_SEEDS)):
    obs = StellaratorEnvironment().reset(seed=seed_idx)
    prompt = render_generation_prompt(obs)
    # Repeat each seed to create a larger training set
    for _ in range(50):
        prompts.append({"prompt": prompt, "seed_idx": seed_idx})

dataset = Dataset.from_list(prompts)
dataset = dataset.shuffle(seed=42)
print(f"Training dataset: {len(dataset)} samples from {len(RESET_SEEDS)} seeds")

## 6. Reward Function

The GRPO training signal comes from **Reward V2**, the environment's built-in reward computed per step in `server/environment.py:_compute_reward_breakdown(...)`. Each generated action plan is rolled out in the local environment, and the cumulative reward across all steps becomes the single scalar GRPO optimizes.

The notebook now uses the same live action contract as the environment itself: plans may include explicit `submit`, and `submit` stays on the same low-fidelity verifier surface as the rest of the episode. Empty or unparseable outputs receive a trainer-side fallback penalty of **−3.0**. Right below, the notebook runs a `submit` smoke so Colab and Northflank confirm the live terminal submit path is wired correctly.

---

### Reward V2 Breakdown

Every step's reward is the sum of the applicable terms below.

#### 1. Step Costs (every non-submit step)

| Term | Value | Condition |
|------|-------|-----------|
| `step_cost` | −0.05 / −0.10 / −0.20 | `run` small / medium / large magnitude |
| `step_cost` | −0.10 | `restore_best` |
| `no_progress_penalty` | −0.20 | `no_progress_steps ≥ 3` (consecutive non-improving steps) |
| `repeat_state_penalty` | −0.15 | Revisiting a previously seen parameter state without improvement |
| `invalid_action_penalty` | −1.0 | `run` action missing parameter, direction, or magnitude |

#### 2. Evaluation Failure

| Term | Value | Condition |
|------|-------|-----------|
| `failure_penalty` | −2.0 | Physics evaluation failed |
| `failure_submit_penalty` | −1.0 | Failed evaluation on `submit` (additional) |
| `failure_budget_penalty` | −0.5 | Failed evaluation on last budget step (additional) |
| `recovery_bonus` | +1.0 | Recovering from a previously failed evaluation |

If evaluation fails, **only** failure terms and step costs apply — the feasibility/objective terms below are skipped.

#### 3. Feasibility Path (constraints NOT all satisfied)

When current or previous state has violated constraints:

| Term | Formula / Value | Purpose |
|------|-----------------|---------|
| `feasibility_crossing_bonus` | +3.0 | Crossing from infeasible → feasible |
| `feasibility_regression_penalty` | −3.0 | Crossing from feasible → infeasible |
| `near_feasible_bonus` | +1.0 | Feasibility dropping below 0.02 threshold |
| `feasibility_delta_reward` | `(prev_feasibility − curr_feasibility) × 2.0` | Progress toward satisfying constraints |
| `best_feasibility_bonus` | `max(0, best_feas_before − curr_feas) × 1.5` | New-best feasibility while still infeasible |
| `triangularity_repair_reward` | `(prev_tri_violation − curr_tri_violation) × 2.0` | Reducing triangularity constraint gap |
| `aspect_ratio_repair_reward` | `(prev_ar_violation − curr_ar_violation) × 1.0` | Reducing aspect ratio constraint gap |
| `iota_repair_reward` | `(prev_iota_violation − curr_iota_violation) × 1.0` | Reducing iota constraint gap |

#### 4. Objective Path (both prev and curr constraints satisfied)

When the design is feasible and stays feasible:

| Term | Formula / Value | Purpose |
|------|-----------------|---------|
| `objective_delta_reward` | `(prev_max_elongation − curr_max_elongation) × 10.0` | Lowering max elongation (the optimization target) |
| `best_score_bonus` | `max(0, curr_score − best_score_before) × 0.75` | New-best P1 score while feasible |

#### 5. Terminal Bonus (on `submit` or final budget step)

| Term | Formula / Value | Condition |
|------|-----------------|-----------|
| `terminal_improvement_bonus` | `5.0 × ratio` (submit) / `2.0 × ratio` (last step) | Feasible and score > initial score |
| `terminal_budget_bonus` | `budget_remaining / budget_total` | Submit only, with improvement |
| `terminal_no_improvement_penalty` | −1.0 (submit) / −0.5 (last step) | No improvement over initial |

Where `ratio = (curr_score − base_score) / max(1.0 − base_score, 1e-6)`.

---

**Constants** (from `server/environment.py`): `FAILURE_PENALTY=−2.0`, `FEASIBILITY_DELTA_WEIGHT=2.0`, `TRIANGULARITY_REPAIR_WEIGHT=2.0`, `ASPECT_RATIO_REPAIR_WEIGHT=1.0`, `IOTA_REPAIR_WEIGHT=1.0`, `BEST_FEASIBILITY_BONUS_WEIGHT=1.5`, `BEST_SCORE_BONUS_WEIGHT=0.75`, `NEAR_FEASIBILITY_THRESHOLD=0.02`, `NEAR_FEASIBILITY_BONUS=1.0`, `NO_PROGRESS_STEP_THRESHOLD=3`, `NO_PROGRESS_PENALTY=−0.2`, `REPEAT_STATE_PENALTY=−0.15`, `RESTORE_STEP_COST=−0.1`, `STEP_COST_BY_MAGNITUDE={small: −0.05, medium: −0.10, large: −0.20}`.

In [ ]:
def environment_reward_fn(
    completions: list[str], seed_idx: list[int] | None = None, **kwargs
) -> list[float]:
    """Execute each action plan in the environment and return cumulative reward.

    This is the sole GRPO training signal in the notebook. It uses the live
    low-fidelity environment reward path and allows explicit submit so the
    trainer stays aligned to the same action contract as the environment.
    Empty or unparseable outputs still receive a trainer-side fallback
    penalty of -3.0. Environment/runtime bugs should raise directly so they
    are not misclassified as bad model outputs.
    """
    rewards = []
    seeds = seed_idx if seed_idx is not None else [0] * len(completions)
    for i, completion in enumerate(completions):
        actions = parse_action_plan(completion)
        if len(actions) == 0:
            rewards.append(-3.0)
            continue
        trace = run_episode_with_actions(
            actions,
            seed_idx=int(seeds[i]) % len(RESET_SEEDS),
        )
        rewards.append(trace.total_reward)
    return rewards


# Test reward function with a hand-crafted plan
test_plan = json.dumps(
    [
        {
            "intent": "run",
            "parameter": "triangularity_scale",
            "direction": "increase",
            "magnitude": "small",
        },
        {
            "intent": "run",
            "parameter": "rotational_transform",
            "direction": "increase",
            "magnitude": "medium",
        },
    ]
)
print(f"Environment reward: {environment_reward_fn([test_plan], seed_idx=[0])}")

# Test terminal submit path on the same live verifier surface
submit_smoke_plan = json.dumps(
    [
        {
            "intent": "run",
            "parameter": "triangularity_scale",
            "direction": "increase",
            "magnitude": "small",
        },
        {"intent": "submit"},
    ]
)
submit_smoke_trace = run_episode_with_actions(
    parse_action_plan(submit_smoke_plan),
    seed_idx=0,
)
final_step = submit_smoke_trace.steps[-1]
if final_step.reward_breakdown.get("intent") != "submit":
    raise RuntimeError("Submit smoke did not end on the terminal submit reward path.")
if final_step.evaluation_fidelity != "low":
    raise RuntimeError(
        f"Expected unified low-fidelity submit path, got {final_step.evaluation_fidelity!r}."
    )
print("Submit smoke confirmed live terminal path:")
print(
    f"  final action={final_step.action_label}, fidelity={final_step.evaluation_fidelity}, "
    f"reward={final_step.reward:+.3f}"
)
print(f"  submit reward terms={final_step.reward_breakdown}")

## 6b. Untrained Model Baseline

Evaluate the base model **before any GRPO training** on all 3 seeds using **greedy decoding** (`do_sample=False`). Greedy decoding is deterministic — the same model + prompt always produces the same output — so the before/after comparison is fully reproducible across reruns.

In [ ]:
MAX_PROMPT_LENGTH = 768
MAX_COMPLETION_LENGTH = MAX_SEQ_LENGTH - MAX_PROMPT_LENGTH

N_RANDOM_ROLLOUTS = 10


def reward_term_summary(step_or_obs: object) -> str:
    """Format non-zero reward terms for display."""
    breakdown_obj = getattr(step_or_obs, "reward_breakdown")
    breakdown = (
        breakdown_obj.model_dump() if hasattr(breakdown_obj, "model_dump") else breakdown_obj
    )
    terms = []
    for key, value in breakdown.items():
        if key in {
            "intent",
            "total",
            "evaluation_failed",
            "recovered_from_failure",
            "reference_constraints_satisfied",
            "reference_score",
            "reference_feasibility",
            "reference_max_elongation",
            "initial_reference_score",
            "terminal_score_ratio",
        }:
            continue
        if isinstance(value, (int, float)) and float(value) != 0.0:
            terms.append(f"{key}={float(value):+.3f}")
    return ", ".join(terms) if terms else "none"


def run_episode_with_model(seed_idx: int) -> tuple[float, list[str], bool]:
    """Run one episode using the current model state (greedy decoding).

    Greedy decoding (do_sample=False) makes the output fully deterministic
    for a given model state and seed, so a single rollout per seed is
    sufficient for reproducible evaluation.
    """
    env = StellaratorEnvironment()
    obs = env.reset(seed=seed_idx)
    prompt = render_generation_prompt(obs)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_COMPLETION_LENGTH,
            do_sample=False,
        )
    completion = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )
    actions = parse_action_plan(completion)
    if len(actions) == 0:
        return -3.0, ["(no valid actions parsed)"], False
    episode = run_episode_with_actions(actions, seed_idx=seed_idx)
    trace = [
        (
            f"{step.action_label} → reward={step.reward:.3f} "
            f"score={step.p1_score:.4f} feasible={step.constraints_satisfied}"
        )
        for step in episode.steps
    ]
    return episode.total_reward, trace, episode.constraints_satisfied


model.eval()
print("=" * 60)
print("UNTRAINED MODEL BASELINE (before GRPO) — greedy, deterministic")
print("=" * 60)
untrained_rewards = []
for seed in range(len(RESET_SEEDS)):
    reward, trace, feasible = run_episode_with_model(seed)
    untrained_rewards.append(reward)
    print(f"\nSeed {seed} — Total reward: {reward:.3f}, Feasible: {feasible}")
    for line in trace:
        print(f"  {line}")

untrained_mean = sum(untrained_rewards) / len(untrained_rewards)
print(f"\nUntrained mean reward: {untrained_mean:.3f}")
print("Snapshot saved. Will compare against trained model after GRPO.")

## 7. GRPO Training

Train the model using Group Relative Policy Optimization. GRPO generates multiple completions per prompt and updates the policy toward higher-reward completions.

In [ ]:
import matplotlib.pyplot as plt

from IPython.display import clear_output, display
from transformers import TrainerCallback
from trl import GRPOConfig, GRPOTrainer


def extract_logged_reward(logs: dict[str, object]) -> float | None:
    reward_value = logs.get("reward")
    if reward_value is None:
        reward_value = logs.get("rewards/environment_reward_fn")
    if isinstance(reward_value, (int, float)):
        return float(reward_value)
    return None


class LiveTrainingMonitorCallback(TrainerCallback):
    def __init__(self, max_steps: int) -> None:
        self.max_steps = max_steps
        self.loss_steps: list[int] = []
        self.losses: list[float] = []
        self.reward_steps: list[int] = []
        self.rewards: list[float] = []

    def _render(self, step: int) -> None:
        clear_output(wait=True)
        latest_loss = self.losses[-1] if self.losses else None
        latest_reward = self.rewards[-1] if self.rewards else None
        best_reward = max(self.rewards) if self.rewards else None
        latest_loss_text = f"{latest_loss:.4f}" if latest_loss is not None else "n/a"
        latest_reward_text = f"{latest_reward:+.4f}" if latest_reward is not None else "n/a"
        best_reward_text = f"{best_reward:+.4f}" if best_reward is not None else "n/a"

        print("GRPO live monitor")
        print(f"step: {step}/{self.max_steps}")
        print(f"latest loss: {latest_loss_text}")
        print(f"latest reward: {latest_reward_text}")
        print(f"best reward so far: {best_reward_text}")

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        if self.losses:
            axes[0].plot(self.loss_steps, self.losses, color="#0b6efd", linewidth=2)
            axes[0].scatter(self.loss_steps[-1], self.losses[-1], color="#0b6efd", s=40)
        else:
            axes[0].text(0.5, 0.5, "waiting for loss logs", ha="center", va="center")
        axes[0].set_xlabel("Step")
        axes[0].set_ylabel("Loss")
        axes[0].set_title("Training Loss")
        axes[0].grid(True, alpha=0.3)

        if self.rewards:
            axes[1].plot(
                self.reward_steps,
                self.rewards,
                color="#198754",
                linewidth=2,
                marker="o",
                markersize=3,
            )
            axes[1].scatter(self.reward_steps[-1], self.rewards[-1], color="#198754", s=40)
        else:
            axes[1].text(0.5, 0.5, "waiting for reward logs", ha="center", va="center")
        axes[1].axhline(0.0, color="0.7", linewidth=1, linestyle="--")
        axes[1].set_xlabel("Step")
        axes[1].set_ylabel("Mean Reward")
        axes[1].set_title("Environment Reward")
        axes[1].grid(True, alpha=0.3)

        fig.suptitle("Fusion Design Lab — Live GRPO Monitor", fontsize=14, fontweight="bold")
        fig.tight_layout()
        display(fig)
        plt.close(fig)

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not state.is_world_process_zero or not logs:
            return

        step = int(state.global_step)
        loss_value = logs.get("loss")
        if isinstance(loss_value, (int, float)):
            if self.loss_steps and self.loss_steps[-1] == step:
                self.losses[-1] = float(loss_value)
            else:
                self.loss_steps.append(step)
                self.losses.append(float(loss_value))

        reward_value = extract_logged_reward(logs)
        if reward_value is not None:
            if self.reward_steps and self.reward_steps[-1] == step:
                self.rewards[-1] = reward_value
            else:
                self.reward_steps.append(step)
                self.rewards.append(reward_value)

        self._render(step)

    def on_train_end(self, args, state, control, **kwargs):
        if state.is_world_process_zero:
            self._render(int(state.global_step))


training_args = GRPOConfig(
    output_dir="./grpo_fusion_output",
    learning_rate=5e-5,
    num_generations=8,
    max_completion_length=MAX_COMPLETION_LENGTH,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    max_steps=60,
    temperature=1.0,
    logging_steps=1,
    save_steps=20,
    bf16=use_bf16,
    fp16=not use_bf16,
    report_to="none",
    seed=42,
)

live_training_callback = LiveTrainingMonitorCallback(max_steps=training_args.max_steps)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[environment_reward_fn],
    args=training_args,
    train_dataset=dataset,
    callbacks=[live_training_callback],
)

print("Starting GRPO training...")
train_result = trainer.train()
print(f"Training complete. Total steps: {train_result.global_step}")

## 8. Training Results

The training cell above renders a live dashboard while GRPO runs. This section saves a clean post-training summary figure.

In [ ]:
log_history = trainer.state.log_history
steps = [entry["step"] for entry in log_history if "loss" in entry]
losses = [entry["loss"] for entry in log_history if "loss" in entry]

# Extract reward metrics if available
reward_steps = [
    entry["step"]
    for entry in log_history
    if "reward" in entry or "rewards/environment_reward_fn" in entry
]
rewards = [
    entry.get("reward", entry.get("rewards/environment_reward_fn", 0))
    for entry in log_history
    if "reward" in entry or "rewards/environment_reward_fn" in entry
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(steps, losses, "b-", alpha=0.7)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("GRPO Training Loss")
axes[0].grid(True, alpha=0.3)

if rewards:
    axes[1].plot(reward_steps, rewards, "g-o", alpha=0.7, markersize=3)
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Mean Reward")
    axes[1].set_title("Environment Reward Over Training")
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "Reward metrics not logged", ha="center", va="center")

plt.suptitle("Fusion Design Lab — GRPO Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved training_curves.png")

## 9. Evaluate Trained Policy

Compare the GRPO-trained model against the **untrained baseline** (captured in Section 6b) and random action selection on the same **live low-fidelity environment contract** used during GRPO, including explicit terminal `submit`.

- **Model evaluations** use deterministic greedy decoding (`do_sample=False`), so results are fully reproducible across reruns. One rollout per seed suffices.
- **Random baseline** remains stochastic, so it averages 10 rollouts per seed for a stable estimate, then explicitly submits its final candidate to stay on the same terminal contract.

In [ ]:
import random

model.eval()

# --- Trained model (greedy = deterministic, 1 rollout per seed) ---
print("=" * 60)
print("TRAINED MODEL (after GRPO) — greedy, deterministic")
print("=" * 60)
trained_rewards = []
for seed in range(len(RESET_SEEDS)):
    reward, trace, feasible = run_episode_with_model(seed)
    trained_rewards.append(reward)
    print(f"\nSeed {seed} — Total reward: {reward:.3f}, Feasible: {feasible}")
    for line in trace:
        print(f"  {line}")

trained_mean = sum(trained_rewards) / len(trained_rewards)

# --- Random baseline (stochastic, averaged over N_RANDOM_ROLLOUTS per seed) ---
print("\n" + "=" * 60)
print(f"RANDOM BASELINE ({N_RANDOM_ROLLOUTS} episodes per seed)")
print("=" * 60)
random_rewards = []
for seed in range(len(RESET_SEEDS)):
    seed_rewards = []
    for _ in range(N_RANDOM_ROLLOUTS):
        random_plan = [
            StellaratorAction(**random.choice(RUN_ACTION_SPECS)) for _ in range(max(BUDGET - 1, 0))
        ]
        random_plan.append(StellaratorAction(intent="submit"))
        seed_rewards.append(
            run_episode_with_actions(
                random_plan,
                seed_idx=seed,
            ).total_reward
        )
    random_rewards.extend(seed_rewards)
    print(
        f"Seed {seed} — Mean: {sum(seed_rewards) / len(seed_rewards):.3f}, "
        f"Best: {max(seed_rewards):.3f}"
    )

random_mean = sum(random_rewards) / len(random_rewards)

# --- Before/After comparison ---
print("\n" + "=" * 60)
print("BEFORE / AFTER COMPARISON")
print("=" * 60)
print(f"  Model evals: greedy (deterministic), 1 rollout × {len(RESET_SEEDS)} seeds")
print(f"  Random baseline: {N_RANDOM_ROLLOUTS} rollouts × {len(RESET_SEEDS)} seeds (averaged)")
print()
print(f"{'Agent':<25} {'Mean Reward':>12}")
print("-" * 39)
print(f"{'Random':<25} {random_mean:>+12.3f}")
print(f"{'Untrained Qwen 3.5-4B':<25} {untrained_mean:>+12.3f}")
print(f"{'GRPO-trained (60 steps)':<25} {trained_mean:>+12.3f}")
print()
improvement = trained_mean - untrained_mean
print(f"GRPO improvement over untrained: {improvement:+.3f}")
print(f"GRPO improvement over random:    {trained_mean - random_mean:+.3f}")

## 10. Connect to Deployed HF Space (Optional)

Demonstrate connecting to the live environment on Hugging Face Spaces through the typed OpenEnv client and running the trained model against it. This section is optional and will skip cleanly if the deployment is unavailable.

**Contract compatibility check:** Before running any episodes, the cell verifies that the remote `/task` endpoint returns the same budget, constraints, parameters, directions, and magnitudes as the local source code. If any field mismatches, the demo is skipped with a diagnostic message — this prevents silent reward or behavior divergence between the notebook and a stale deployment.

In [ ]:
import requests

from fusion_lab.client import FusionLabClient
from server.physics import (
    ASPECT_RATIO_MAX,
    AVERAGE_TRIANGULARITY_MAX,
    EDGE_IOTA_OVER_NFP_MIN,
)

HF_SPACE_URL = "https://creativeengineer-fusion-design-lab.hf.space"
REQUEST_TIMEOUT_SECONDS = 10

try:
    health_response = requests.get(f"{HF_SPACE_URL}/health", timeout=REQUEST_TIMEOUT_SECONDS)
except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as exc:
    health_response = None
    print(f"Skipping remote demo — network error reaching HF Space: {exc}")

if health_response is not None and health_response.status_code != 200:
    print(
        "Skipping remote demo because the HF Space is unavailable: "
        f"/health returned {health_response.status_code}."
    )
    health_response = None

if health_response is not None:
    health = health_response.json()
    print(f"HF Space status: {health['status']}")

    try:
        task_response = requests.get(f"{HF_SPACE_URL}/task", timeout=REQUEST_TIMEOUT_SECONDS)
    except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as exc:
        task_response = None
        print(f"Skipping remote demo — network error reaching /task: {exc}")

    if task_response is not None and task_response.status_code != 200:
        print(
            "Skipping remote demo because the HF Space task endpoint is unavailable: "
            f"/task returned {task_response.status_code}."
        )
        task_response = None

    # ── Contract compatibility check ──────────────────────────────────────
    if task_response is not None:
        task = task_response.json()
        expected_contract = {
            "budget": BUDGET,
            "constraints": {
                "aspect_ratio_max": ASPECT_RATIO_MAX,
                "average_triangularity_max": AVERAGE_TRIANGULARITY_MAX,
                "abs_edge_iota_over_nfp_min": EDGE_IOTA_OVER_NFP_MIN,
            },
            "parameters": list(RUN_PARAMETERS),
            "directions": list(RUN_DIRECTIONS),
            "magnitudes": list(RUN_MAGNITUDES),
        }
        mismatches: list[str] = []
        for key in expected_contract:
            remote_val = task.get(key)
            local_val = expected_contract[key]
            if remote_val != local_val:
                mismatches.append(f"  {key}: local={local_val!r}  remote={remote_val!r}")

        if mismatches:
            print("Skipping remote demo — contract mismatch between local code and HF Space:")
            for m in mismatches:
                print(m)
            print("Redeploy the Space with the current code to fix this.")
            task_response = None
        else:
            print("Contract check passed — remote matches local code.")
            print(f"\nTask: {task['description']}")
            print(f"Constraints: {task['constraints']}")
            print(f"Budget: {task['budget']}")

    # ── Run trained model against remote environment ──────────────────────
    if task_response is not None:
        with FusionLabClient(base_url=HF_SPACE_URL) as env:
            reset_result = env.reset(seed=42)
            remote_obs = reset_result.observation
            print(f"\nRemote reset — max_elongation: {remote_obs.max_elongation:.4f}")
            print(f"  aspect_ratio: {remote_obs.aspect_ratio:.4f}")
            print(f"  constraints_satisfied: {remote_obs.constraints_satisfied}")
            print(f"  budget_remaining: {remote_obs.budget_remaining}")

            prompt = render_generation_prompt(remote_obs)
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs, max_new_tokens=MAX_COMPLETION_LENGTH, do_sample=False
                )
            completion = tokenizer.decode(
                outputs[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
            )
            actions = parse_action_plan(completion)

            print(f"\nTrained model generated {len(actions)} actions for remote env:")
            for i, action in enumerate(actions[:BUDGET], start=1):
                result = env.step(action)
                step_obs = result.observation
                reward = float(result.reward) if result.reward is not None else 0.0
                print(
                    f"  Step {i}: {action.intent} {action.parameter or ''} "
                    f"{action.direction or ''} {action.magnitude or ''} "
                    f"→ reward={reward:.3f}, score={step_obs.p1_score:.4f}, "
                    f"terms={reward_term_summary(step_obs)}"
                )
                if result.done:
                    print(f"  Episode done. Final score: {step_obs.p1_score:.4f}")
                    break
        print("\nEnvironment is live and accessible for training and evaluation.")